![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 06: Advanced)**

**Session 6A: IBM Differential Privacy Library in 30 Seconds**

---

- Materials in this module have been developed to support practical learning in modern data science, privacy-aware analytics, and applied machine learning.
- Materials may include adapted or referenced open-source resources. Keep attribution and licence notes where applicable.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find any issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This notebook is one component of the practical and self-learning materials for Module 06.</td>
</tr>
<tr>
<td align="left">Estimated duration</td>
<td>Approximately 48 minutes, based on 240 minutes of M06 practical/self-learning work across five notebooks.</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter with Python packages available.</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>A differentially private Gaussian Naive Bayes model and an epsilon-accuracy comparison.</td>
</tr>
<tr>
<td align="left">Related assessment</td>
<td>General practical skill development; not directly assessed.</td>
</tr>
</tbody>
</table>

</div>

---

**Table of Contents**

- [1. Overview and Learning Goals](#1-overview-and-learning-goals)
- [2. Setup and Required Packages](#2-setup-and-required-packages)
- [3. Background Concepts](#3-background-concepts)
- [4. Guided Example: Private Naive Bayes on Iris](#4-guided-example-private-naive-bayes-on-iris)
- [5. Practical Exercise: Vary the Privacy Budget](#5-practical-exercise-vary-the-privacy-budget)
- [6. Student Tasks](#6-student-tasks)
- [7. Checks, Reflection, and References](#7-checks-reflection-and-references)

<a id="1-overview-and-learning-goals"></a>

### 1. Overview and Learning Goals

Differential privacy adds controlled randomness to a data-analysis or machine-learning workflow so that the output reveals less about any one training record. In this practical you will use the IBM Differential Privacy Library, `diffprivlib`, to train a private Gaussian Naive Bayes classifier on the Iris dataset.

The notebook keeps the original quick-start idea: load a familiar dataset, fit a differentially private classifier, and compare accuracy across several `epsilon` values.

By the end of this practical, you should be able to:

1. install and import `diffprivlib` in an online or local notebook environment;
2. train a differentially private classifier using an `sklearn`-style API;
3. explain why feature bounds and `epsilon` matter for private learning;
4. compare model accuracy across a range of privacy budgets;
5. describe the privacy-utility trade-off in plain language.

<a id="2-setup-and-required-packages"></a>

### 2. Setup and Required Packages

This notebook uses the built-in Iris dataset from `scikit-learn`; no external data file is required.

#### Option A: Google Colab / online execution

Keep `EXECUTION_MODE = "online"`. The setup cell installs missing packages into the temporary notebook runtime if needed.

#### Option B: Local repository execution

Use `EXECUTION_MODE = "local"` only when your local Python environment already has the required packages installed. Local mode raises a clear error instead of changing your environment automatically.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

EXECUTION_MODE = "online"  # Use "online" for Google Colab; use "local" for a managed local environment.
INSTALL_MISSING_PACKAGES = EXECUTION_MODE == "online"
RANDOM_STATE = 742

PACKAGE_SPECS = [
    "diffprivlib",
    "scikit-learn<1.8",
    "matplotlib",
    "pandas",
]


def version_prefix(version, length=2):
    parts = []
    for part in version.split(".")[:length]:
        digits = "".join(char for char in part if char.isdigit())
        parts.append(int(digits) if digits else 0)
    while len(parts) < length:
        parts.append(0)
    return tuple(parts)


def scikit_learn_is_compatible():
    if importlib.util.find_spec("sklearn") is None:
        return False
    import sklearn
    return version_prefix(sklearn.__version__) < (1, 8)


def ensure_packages_available():
    import_map = {
        "diffprivlib": "diffprivlib",
        "sklearn": "scikit-learn<1.8",
        "matplotlib": "matplotlib",
        "pandas": "pandas",
    }
    missing = [
        pip_name
        for import_name, pip_name in import_map.items()
        if importlib.util.find_spec(import_name) is None
    ]
    incompatible = []
    if not scikit_learn_is_compatible():
        incompatible.append("scikit-learn<1.8")
    if not missing and not incompatible:
        return
    if INSTALL_MISSING_PACKAGES:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGE_SPECS])
        return
    raise ModuleNotFoundError(
        "Missing or incompatible required packages: "
        + ", ".join(sorted(set(missing + incompatible)))
        + ". Install them in the active environment, or set EXECUTION_MODE = 'online' to install them in an online notebook runtime."
    )


ensure_packages_available()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from diffprivlib.models import GaussianNB

print("Setup complete.")
print("Current working directory:", Path.cwd())

<a id="3-background-concepts"></a>

### 3. Background Concepts

`diffprivlib` provides differentially private versions of familiar analytics and machine-learning tools.

Key terms for this practical:

- **Differential privacy**: a formal privacy guarantee that limits how much the output can change when one record is added or removed.
- **Privacy budget (`epsilon`)**: a positive value controlling the privacy-utility trade-off. Smaller values usually mean stronger privacy and noisier outputs.
- **Bounds**: public feature limits used to calibrate privacy-preserving noise. Bounds should be specified rather than learned from sensitive data.
- **Utility**: how useful the output remains for the learning task, here measured using test accuracy.

<a id="4-guided-example-private-naive-bayes-on-iris"></a>

### 4. Guided Example: Private Naive Bayes on Iris

First load the Iris dataset and create a reproducible 80/20 train-test split. The Iris dataset is a teaching dataset with four numeric flower measurements and three species labels.

In [ ]:
iris = datasets.load_iris()

X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.2,
    stratify=iris.target,
    random_state=RANDOM_STATE,
)

feature_table = pd.DataFrame(iris.data, columns=iris.feature_names)
print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
feature_table.head()

The private classifier follows the familiar `sklearn` pattern: create the model, call `.fit()`, then call `.predict()` or `.score()`.

The important privacy-specific parameters are:

- `bounds`: public feature limits used by the mechanism;
- `epsilon`: the privacy budget for this model fit;
- `random_state`: a reproducibility setting for the teaching example.

In [ ]:
iris_bounds = ([4.3, 2.0, 1.1, 0.1], [7.9, 4.4, 6.9, 2.5])

private_nb = GaussianNB(bounds=iris_bounds, epsilon=1.0, random_state=RANDOM_STATE)
private_nb.fit(X_train, y_train)

predictions = private_nb.predict(X_test)
test_accuracy = private_nb.score(X_test, y_test)

pd.DataFrame({
    "predicted_label": predictions[:10],
    "actual_label": y_test[:10],
})

The model can now score unseen test examples. Because the training algorithm is private, rerunning with different random seeds or privacy budgets can change the result.

In [ ]:
print(f"Test accuracy with epsilon=1.0: {test_accuracy:.3f}")

<a id="5-practical-exercise-vary-the-privacy-budget"></a>

### 5. Practical Exercise: Vary the Privacy Budget

The next cell trains the same private model across a range of `epsilon` values. Use the plot to reason about the trade-off between privacy and predictive utility.

In [ ]:
epsilons = np.logspace(-2, 2, 30)
accuracy_records = []

for index, epsilon in enumerate(epsilons):
    model = GaussianNB(
        bounds=iris_bounds,
        epsilon=float(epsilon),
        random_state=RANDOM_STATE + index,
    )
    model.fit(X_train, y_train)
    accuracy_records.append({
        "epsilon": float(epsilon),
        "accuracy": model.score(X_test, y_test),
    })

epsilon_results = pd.DataFrame(accuracy_records)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(epsilon_results["epsilon"], epsilon_results["accuracy"], marker="o")
ax.set_title("Differentially private Naive Bayes accuracy")
ax.set_xlabel("epsilon")
ax.set_ylabel("test accuracy")
ax.grid(True, which="both", alpha=0.3)
plt.show()

epsilon_results.head()

Try a small set of your own values. Keep the values positive. Then compare whether the results match your expectation about stronger or weaker privacy.

In [ ]:
# Student workspace
# Change these epsilon values and rerun the cell. Smaller epsilon usually means
# stronger privacy and more noise; larger epsilon usually means weaker privacy
# and less noise.
student_epsilons = [0.05, 0.5, 5.0]

student_records = []
for index, epsilon in enumerate(student_epsilons):
    model = GaussianNB(
        bounds=iris_bounds,
        epsilon=float(epsilon),
        random_state=RANDOM_STATE + 100 + index,
    )
    model.fit(X_train, y_train)
    student_records.append({
        "epsilon": epsilon,
        "accuracy": model.score(X_test, y_test),
    })

student_results = pd.DataFrame(student_records)
student_results

<a id="6-student-tasks"></a>

### 6. Student Tasks

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Task 1</td>
<td>Run the notebook with three different `epsilon` values.</td>
<td>`epsilon` controls the privacy-utility trade-off.</td>
<td>A small table of epsilon and accuracy values.</td>
</tr>
<tr>
<td align="left">Task 2</td>
<td>Explain why the `bounds` values should be treated as public knowledge.</td>
<td>Bounds learned from sensitive data can leak information.</td>
<td>A short written explanation in a markdown cell.</td>
</tr>
<tr>
<td align="left">Task 3</td>
<td>Compare the private model result with your expectation from a non-private classifier.</td>
<td>Private learning should be judged against a clear utility baseline.</td>
<td>A brief comparison of accuracy and privacy trade-off.</td>
</tr>
</tbody>
</table>

</div>

<a id="7-checks-reflection-and-references"></a>

### 7. Checks, Reflection, and References

Use these checks to confirm that the notebook state is coherent before interpreting the privacy-utility plot.

In [ ]:
assert X_train.shape[0] == 120
assert X_test.shape[0] == 30
assert predictions.shape == y_test.shape
assert epsilon_results["accuracy"].between(0, 1).all()

print("Checks passed.")
print(f"Baseline epsilon: 1.0")
print(f"Baseline test accuracy: {test_accuracy:.3f}")
print("Compared epsilon values:", len(epsilon_results))

Reflection questions:

1. What happened to test accuracy when `epsilon` was very small?
2. Why does specifying public feature bounds matter for differential privacy?
3. In a real data project, who should decide the privacy budget?

Further reading:

- [IBM Differential Privacy Library](https://github.com/IBM/differential-privacy-library)
- [diffprivlib model documentation](https://diffprivlib.readthedocs.io/en/latest/modules/models.html)
- [scikit-learn Iris dataset](https://scikit-learn.org/stable/auto_examples/datasets/plot_iris_dataset.html)

Dataset note: this notebook uses the built-in Iris dataset from `scikit-learn` for teaching purposes.